In [1]:
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

# This is required to run multiple processes with JAX.
from multiprocessing import set_start_method
set_start_method('forkserver', force=True)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from tqdm import tqdm
from pathlib import Path

from config import Config, AssimConfig
import data
from train import Trainer

In [3]:
# Build a base model

cfg = Config.from_file("/nas/cee-water/cjgleason/ted/swot-ml/runs/test/era5.yml")
cfg.quiet = False
mgr = data.DynamicCacheManager(cfg)
cache_dir = mgr.create_cache('train')
dataset = data.CachedBasinGraphDataset(cfg, cache_dir, 'train')
train_dl = dataloader = data.CachedBasinGraphDataLoader(cfg, dataset)

trainer = Trainer(cfg, train_dl)
trainer.save_state()

Caches will be stored at: /scratch3/workspace/tlanghorst_umass_edu-swot-ml-zarr2/20251004/_cache2/920ad311aff6af71/<subset>


Loading basins:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating training statistics for encoding and normalization...


Loading basins: 100%|██████████| 2/2 [00:01<00:00,  1.03it/s]


Wrote 2 new basin files to cache.
✅ Cached resources are loaded and ready.
Calculating training statistics for encoding and normalization...
Loading static attributes...Done!
Loading basin graphs...Done!


Building sampling index: 100%|██████████| 2/2 [00:00<00:00, 10.31it/s]
/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Dataloader using 12 parallel CPU worker(s).
Model contains 465,997 parameters
Logging at /nas/cee-water/cjgleason/ted/swot-ml/runs/test/era5_20251027_132756
2025-10-27 13:27:56,578 - INFO - Saving model checkpoint at step 000000. Average loss since last checkpoint: nan


/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [4]:
# Add an assimilation
trainer = trainer.load_last_checkpoint(trainer.log_dir)

assim_path = Path("/nas/cee-water/cjgleason/ted/swot-ml/runs/_assim/swot_large.yml")
assim_cfg = AssimConfig.from_file(assim_path)
trainer.cfg.add_assimilation(assim_cfg)
cfg = trainer.cfg

# Update the logging
now_str = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
assim_log_dir = trainer.log_dir / f"{assim_path.stem}_{now_str}"
trainer.new_logger(assim_log_dir)
# Separates the copied training log from the new messages.
new_feats = list(assim_cfg.new_features.keys())
log_msg = f"{'=' * 10} Starting new assimilation: {new_feats} {'=' * 10}"
trainer.logger.info("=" * len(log_msg))
trainer.logger.info(log_msg)
trainer.logger.info("=" * len(log_msg))

# Create a new dataloader and update in trainer
manager = data.DynamicCacheManager(cfg)
cache_dir = manager.create_cache("train")
dataset = data.CachedBasinGraphDataset(cfg, cache_dir, "train")
trainer.training_dl = data.CachedBasinGraphDataLoader(cfg, dataset)


# Update the model with the new assimilation modules
new_model = trainer.model
for new_group in assim_cfg.new_features.keys():
    n_group_features = len(dataset.features.dynamic[new_group])
    group_network_sizes = assim_cfg.model_args.get(new_group, {})
    new_model.add_assimilator(new_group, n_group_features, group_network_sizes)
    trainer.cfg.model_args.sparse_sizes[new_group] = n_group_features
trainer.replace_model(new_model)

    
trainer.save_state()

Model contains 465,997 parameters
Loading trainer from checkpoint step_000000
Logging at /nas/cee-water/cjgleason/ted/swot-ml/runs/test/era5_20251027_132756
Logging at /nas/cee-water/cjgleason/ted/swot-ml/runs/test/era5_20251027_132756/swot_large_20251027_132757
2025-10-27 13:27:57,421 - INFO - ===============================================================
2025-10-27 13:27:57,422 - INFO - ========== Starting new assimilation: ['swot-river'] ==========
2025-10-27 13:27:57,422 - INFO - ===============================================================


/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/pydantic/main.py:519: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `int` - serialized value may not be as expected [input_value=inf, input_type=float])
  return self.__pydantic_serializer__.to_json(


Caches will be stored at: /scratch3/workspace/tlanghorst_umass_edu-swot-ml-zarr2/20251004/_cache2/634dba7e102e12e3/<subset>


Loading basins:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating training statistics for encoding and normalization...


Loading basins: 100%|██████████| 2/2 [00:03<00:00,  1.71s/it]


Wrote 2 new basin files to cache.
✅ Cached resources are loaded and ready.
Calculating training statistics for encoding and normalization...
Loading static attributes...Done!
Loading basin graphs...Done!


Building sampling index: 100%|██████████| 2/2 [00:00<00:00, 10.15it/s]
/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Dataloader using 12 parallel CPU worker(s).
2025-10-27 13:28:05,386 - INFO - Saving model checkpoint at step 000000. Average loss since last checkpoint: nan


/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/pydantic/main.py:519: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `int` - serialized value may not be as expected [input_value=inf, input_type=float])
  return self.__pydantic_serializer__.to_json(


In [15]:
# trainer.opt_state[0]

In [16]:
# trainer.model.sparse_embedders

In [5]:
trainer.log_dir

PosixPath('/nas/cee-water/cjgleason/ted/swot-ml/runs/test/era5_20251027_132756/swot_large_20251027_132757')

In [5]:
checkpoint_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/runs/test/era5_20251024_155317/swot_large_20251024_155318/checkpoints/step_000000")
load_cfg = Config.from_file(checkpoint_dir / 'config.json')

In [8]:
import jax
import jax.numpy as jnp
import equinox as eqx
from collections.abc import Mapping

def compare_pytrees(tree1, tree2, path=()):
    """Recursively compare two PyTrees and report mismatches."""
    differences = []

    # Handle dicts and objects with attributes
    if isinstance(tree1, Mapping) and isinstance(tree2, Mapping):
        keys1, keys2 = set(tree1.keys()), set(tree2.keys())
        for k in keys1 - keys2:
            differences.append((path + (k,), "extra in tree1"))
        for k in keys2 - keys1:
            differences.append((path + (k,), "extra in tree2"))
        for k in keys1 & keys2:
            differences += compare_pytrees(tree1[k], tree2[k], path + (k,))

    # Handle Equinox/nn.Module-like objects (with __dict__)
    elif hasattr(tree1, "__dict__") and hasattr(tree2, "__dict__"):
        return compare_pytrees(vars(tree1), vars(tree2), path)

    # Handle lists/tuples
    elif isinstance(tree1, (list, tuple)) and isinstance(tree2, (list, tuple)):
        if len(tree1) != len(tree2):
            differences.append((path, f"length mismatch {len(tree1)} vs {len(tree2)}"))
        for i, (a, b) in enumerate(zip(tree1, tree2)):
            differences += compare_pytrees(a, b, path + (i,))

    # Handle JAX arrays
    elif isinstance(tree1, jnp.ndarray) and isinstance(tree2, jnp.ndarray):
        if tree1.shape != tree2.shape:
            differences.append((path, f"shape mismatch {tree1.shape} vs {tree2.shape}"))
        elif tree1.dtype != tree2.dtype:
            differences.append((path, f"dtype mismatch {tree1.dtype} vs {tree2.dtype}"))

    # Handle None or other leafs
    elif type(tree1) != type(tree2):
        differences.append((path, f"type mismatch {type(tree1)} vs {type(tree2)}"))

    return differences

diffs = compare_pytrees(loaded_model, trainer.model)
for path, msg in diffs:
    print(" -> ".join(map(str, path)), ":", msg)

In [7]:
loaded_model.sparse_sources

['swot-river']

2025-10-24 15:06:32,373 - INFO - Saving model checkpoint at step 000000. Average loss since last checkpoint: nan


/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [9]:
# Load from file 
trainer = Trainer.load_from_run_or_state(trainer.log_dir)

Model contains 522,509 parameters
Loading trainer from checkpoint step_000000
Logging at /nas/cee-water/cjgleason/ted/swot-ml/runs/test/era5_20251024_160140/swot_large_20251024_160140


In [11]:
cfg.model_args.assim_sizes

{'swot-river': {'embed_width': 96,
  'embed_depth': 2,
  'gain_width': 96,
  'gain_depth': 2}}

In [12]:
# Update the logging
now_str = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
assim_log_dir = trainer.log_dir / f"{assim_path.stem}_{now_str}"
trainer.new_logger(assim_log_dir)
# Separates the copied training log from the new messages.
new_feats = list(assim_cfg.new_features.keys())
log_msg = f"{'=' * 10} Starting new assimilation: {new_feats} {'=' * 10}"
trainer.logger.info("=" * len(log_msg))
trainer.logger.info(log_msg)
trainer.logger.info("=" * len(log_msg))

Logging at /nas/cee-water/cjgleason/ted/swot-ml/runs/test/era5_20251024_150632/swot_large_20251024_150638
2025-10-24 15:06:38,638 - INFO - ===============================================================
2025-10-24 15:06:38,639 - INFO - ========== Starting new assimilation: ['swot-river'] ==========
2025-10-24 15:06:38,640 - INFO - ===============================================================


/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/pydantic/main.py:519: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `int` - serialized value may not be as expected [input_value=inf, input_type=float])
  return self.__pydantic_serializer__.to_json(


In [13]:
trainer.cfg.model_args.assim_sizes

{'swot-river': {'embed_width': 96,
  'embed_depth': 2,
  'gain_width': 96,
  'gain_depth': 2}}

In [36]:
print(trainer.opt_state[0].mu.sparse_embedders)

{'swot-river': StaticMLP(
  append_static=None,
  mlp=MLP(
    layers=(
      Linear(
        weight=f32[96,133],
        bias=f32[96],
        in_features=133,
        out_features=96,
        use_bias=True
      ),
      Linear(
        weight=f32[96,96],
        bias=f32[96],
        in_features=96,
        out_features=96,
        use_bias=True
      ),
      Linear(
        weight=f32[64,96],
        bias=f32[64],
        in_features=96,
        out_features=64,
        use_bias=True
      )
    ),
    activation=None,
    final_activation=None,
    use_bias=True,
    use_final_bias=True,
    in_size=133,
    out_size=64,
    width_size=96,
    depth=2
  )
)}


In [14]:
# Create a new dataloader and update in trainer
manager = data.DynamicCacheManager(trainer.cfg)
cache_dir = manager.create_cache("train")
dataset = data.CachedBasinGraphDataset(trainer.cfg, cache_dir, "train")
trainer.training_dl = data.CachedBasinGraphDataLoader(trainer.cfg, dataset)

Caches will be stored at: /scratch3/workspace/tlanghorst_umass_edu-swot-ml-zarr2/20251004/_cache2/634dba7e102e12e3/<subset>


Loading basins: 100%|██████████| 2/2 [00:00<00:00, 2593.08it/s]

Wrote 0 new basin files to cache.
✅ Cached resources are loaded and ready.
Calculating training statistics for encoding and normalization...
Loading static attributes...

Done!
Loading basin graphs...Done!


Building sampling index: 100%|██████████| 2/2 [00:00<00:00,  7.79it/s]

Dataloader using 12 parallel CPU worker(s).



/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [15]:
# Update the model with the new assimilation modules
new_model = trainer.model
for new_grp in assim_cfg.new_features.keys():
    n_group_features = len(dataset.features.dynamic[new_grp])
    group_network_sizes = assim_cfg.model_args.get(new_grp, {})
    new_model.add_assimilator(new_grp, n_group_features, group_network_sizes)
    trainer.cfg.model_args.sparse_sizes[new_grp] = n_group_features
    print(f"{new_grp}: {n_group_features}")

swot-river: 32


In [17]:
trainer.replace_model(new_model)

In [27]:
trainer.save_state()

2025-10-24 15:11:03,644 - INFO - Saving model checkpoint at step 000000. Average loss since last checkpoint: nan


/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/pydantic/main.py:519: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `int` - serialized value may not be as expected [input_value=inf, input_type=float])
  return self.__pydantic_serializer__.to_json(


In [30]:
trainer.cfg.model_args.assim_sizes

{'swot-river': {'embed_width': 96,
  'embed_depth': 2,
  'gain_width': 96,
  'gain_depth': 2}}

In [28]:
trainer.log_dir

PosixPath('/nas/cee-water/cjgleason/ted/swot-ml/runs/test/era5_20251024_150632/swot_large_20251024_150638')

In [29]:
loaded_assim_trainer = Trainer.load_last_checkpoint(trainer.log_dir)

Model contains 522,509 parameters
Loading trainer from checkpoint step_000000
Logging at /nas/cee-water/cjgleason/ted/swot-ml/runs/test/era5_20251024_150632/swot_large_20251024_150638


In [29]:
loaded_assim_trainer.cfg.model_args.assim_sizes

{'swot-river': {'embed_width': 96,
  'embed_depth': 2,
  'gain_width': 96,
  'gain_depth': 2}}

In [31]:
cfg = Config.from_file("/nas/cee-water/cjgleason/ted/swot-ml/runs/test/era5_20251024_143128/swot_large_20251024_143735/checkpoints/step_000300/config.json")

cfg.model_args

STGATArgs(hidden_size=64, dropout=0.3, head='linear', seq2seq=True, seed=0, name='st_gat', k_hops=3, num_heads=2, return_weights=False, target=['discharge'], seq_length=90, dense_sizes={'era5': 10}, sparse_sizes={'swot-river': 32}, static_size=101, assim_sizes={'swot-river': {'embed_width': 96, 'embed_depth': 2, 'gain_width': 96, 'gain_depth': 2}})

In [33]:
manager = data.DynamicCacheManager(cfg)
cache_dir = manager.create_cache('test')
dataset = data.CachedBasinGraphDataset(cfg, cache_dir, 'test')
dataloader = data.CachedBasinGraphDataLoader(cfg, dataset)


Caches will be stored at: /scratch3/workspace/tlanghorst_umass_edu-swot-ml-zarr2/20251004/_cache2/634dba7e102e12e3/<subset>
Calculating training statistics for encoding and normalization...
Wrote 2 new basin files to cache.
✅ Cached resources are loaded and ready.
Calculating training statistics for encoding and normalization...
Loading static attributes...Done!
Loading basin graphs...Done!
Dataloader using 12 parallel CPU worker(s).


/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [36]:
import models
model_cfg, model = models.make(cfg)

Model contains 522,509 parameters


In [41]:
model.sparse_embedders

{'swot-river': StaticMLP(
   append_static=True,
   mlp=MLP(
     layers=(
       Linear(
         weight=f32[96,133],
         bias=f32[96],
         in_features=133,
         out_features=96,
         use_bias=True
       ),
       Linear(
         weight=f32[96,96],
         bias=f32[96],
         in_features=96,
         out_features=96,
         use_bias=True
       ),
       Linear(
         weight=f32[64,96],
         bias=f32[64],
         in_features=96,
         out_features=64,
         use_bias=True
       )
     ),
     activation=<PjitFunction of <function relu at 0x70e1711a77e0>>,
     final_activation=<function <lambda>>,
     use_bias=True,
     use_final_bias=True,
     in_size=133,
     out_size=64,
     width_size=96,
     depth=2
   )
 )}

In [12]:
trainer.load_last_checkpoint(Path('/nas/cee-water/cjgleason/ted/swot-ml/runs/test/era5_20251024_155317/swot_large_20251024_155318'))

Model contains 522,509 parameters
Loading trainer from checkpoint step_000000
Logging at /nas/cee-water/cjgleason/ted/swot-ml/runs/test/era5_20251024_155317/swot_large_20251024_155318


In [7]:
from config import Config
import models
import train
import equinox as eqx
import optax

cfg = Config.from_file("/nas/cee-water/cjgleason/ted/swot-ml/runs/test/era5_20251024_143128/swot_large_20251024_143735/checkpoints/step_000300/config.json")

# --- Load Model and Optimizer State ---
with open("/nas/cee-water/cjgleason/ted/swot-ml/runs/test/era5_20251024_143128/swot_large_20251024_143735/checkpoints/step_000300/model_and_opt.eqx", "rb") as f:
    _, serialized_model = models.make(cfg)
    
    # # Ensure all leaves are jnp float 32s.
    # # Bandaid for some poorly specified graph adjacency matrices
    # serialized_model = jax.tree_util.tree_map(
    #     lambda x: (jnp.array(x) if isinstance(x, np.ndarray) else x),
    #     serialized_model,
    # )
    model = eqx.tree_deserialise_leaves(f, serialized_model)

    optim = optax.adam(train.trainer._create_lr_schedule(cfg))
    serialized_opt_state = optim.init(eqx.filter(model, eqx.is_inexact_array))
    opt_state = eqx.tree_deserialise_leaves(f, serialized_opt_state)

Model contains 522,509 parameters


TreePathError: Error at leaf with path (SequenceKey(idx=0), GetAttrKey(name='mu'), GetAttrKey(name='spat_temp_lstm'), GetAttrKey(name='gat'), GetAttrKey(name='fwd_layers'), SequenceKey(idx=2), GetAttrKey(name='output_proj'), GetAttrKey(name='weight'))

In [15]:
model.sparse_embedders

{'swot-river': StaticMLP(
   append_static=True,
   mlp=MLP(
     layers=(
       Linear(
         weight=f32[96,133],
         bias=f32[96],
         in_features=133,
         out_features=96,
         use_bias=True
       ),
       Linear(
         weight=f32[96,96],
         bias=f32[96],
         in_features=96,
         out_features=96,
         use_bias=True
       ),
       Linear(
         weight=f32[64,96],
         bias=f32[64],
         in_features=96,
         out_features=64,
         use_bias=True
       )
     ),
     activation=<PjitFunction of <function relu at 0x7916d3441760>>,
     final_activation=<function <lambda>>,
     use_bias=True,
     use_final_bias=True,
     in_size=133,
     out_size=64,
     width_size=96,
     depth=2
   )
 )}